# Practice 28 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: `.cat.remove_unused_categories()`

When you filter a categorical Series, the **unused categories don't disappear automatically** — they stay in `.cat.categories` even if no rows have that value.

```python
s = pd.Categorical(['a', 'b', 'a'], categories=['a', 'b', 'c'])
s.remove_unused_categories()  # → categories=['a', 'b']  ('c' is gone)
```

On a Series:
```python
df['col'].cat.remove_unused_categories()
```

This is useful after filtering — otherwise groupby and value_counts still show empty categories.

---
### Task 1: penguins — category cleanup after filtering

Load `sns.load_dataset('penguins')`.

1. Check memory usage (deep=True). Then convert `species`, `island`, and `sex` to `category`. How much memory did you save in total?
2. Filter to only Biscoe island penguins. Call `.value_counts()` on the `species` column of your filtered DataFrame — notice it still shows all 3 species. Fix this using `.cat.remove_unused_categories()`.
3. Use a list comprehension to collect all column names that are currently `category` dtype.
4. Print `.cat.codes` for the first 5 rows of `island` (on the full DataFrame). What integer does each island map to?

In [82]:
# Your code here
penguins = sns.load_dataset('penguins')
penguins.memory_usage(deep = True)
penguins_2 = penguins.copy()
penguins_2[['species','island','sex']] = penguins_2[['species','island','sex']].astype('category')

penguins_2.memory_usage(deep=True)

### the original is already in category type 

penguins_2.memory_usage(deep=True).sum() - penguins.memory_usage(deep=True).sum()


np.int64(-62712)

In [83]:
penguins.dtypes

species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object

In [39]:
f = penguins[penguins['island'] == 'Biscoe'].copy()
f['species'].value_counts()
f['species'] = f['species'].cat.remove_unused_categories()
f['species'].value_counts()

cn = [col for col in penguins.columns if penguins[col].dtype == 'category']
cn

penguins_2.head(5)['island'].cat.codes

penguins_2['island'].cat.categories

mdict = {cat: code for code, cat in enumerate(penguins_2['island'].cat.categories)}
mdict

{'Biscoe': 0, 'Dream': 1, 'Torgersen': 2}

---
## Level 2 — Transformations

### Task 2: diamonds — answer these questions

Load `sns.load_dataset('diamonds')`. Apply an ordered `CategoricalDtype` to `cut` (`Fair < Good < Very Good < Premium < Ideal`) and to `color` — `color` runs `D < E < F < G < H < I < J` where D is the best (most colorless).

Answer these four questions using whatever approach you prefer — no sub-steps:

1. What percentage of diamonds are cut `'Premium'` or better?
2. Within each cut level, what's the average price? Use NumPy, not groupby.
3. Add a `price_per_carat` column. Which cut has the highest median price_per_carat?
4. Build a summary: mean `price_per_carat` for each `(cut, color)` combination. Stack it so it's a Series with a MultiIndex, then use `.xs()` to pull out all rows for `'Ideal'` cut.

In [86]:
# Your code here
diamonds = sns.load_dataset('diamonds')
cuto = pd.CategoricalDtype(categories=['Fair','Good','Very Good','Premium','Ideal'],
                         ordered=True)

diamonds['cut'] = diamonds['cut'].astype(cuto)

coloro = pd.CategoricalDtype(categories=['D','E','F','G','H','I','J'],
                             ordered=True)
diamonds['color'] = diamonds['color'].astype(coloro)

print((diamonds['cut']>='Premium').mean())

cut_avg = {}
for cut in diamonds['cut'].unique():
    cut_avg[cut] = diamonds[diamonds['cut'] == cut]['price'].mean()

cut_avg


for cut in diamonds['cut'].cat.categories:
    print(cut, np.mean(diamonds.loc[diamonds['cut']==cut,'price']))
    print(cut, np.mean(diamonds[diamonds['cut']==cut]['price']))


diamonds['price_per_carat'] = diamonds['price']/diamonds['carat']
diamonds.groupby('cut', observed=True)['price_per_carat'].agg('median').idxmax()

p = diamonds.pivot_table(
    index='cut',
    columns='color',
    values = 'price_per_carat'
).stack()

p.xs('Ideal',level ='cut')



0.6552094920281795
Fair 4358.757763975155
Fair 4358.757763975155
Good 3928.864451691806
Good 3928.864451691806
Very Good 3981.7598907465654
Very Good 3981.7598907465654
Premium 4584.2577042999055
Premium 4584.2577042999055
Ideal 3457.541970210199
Ideal 3457.541970210199


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_47706/2335568122.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  p = diamonds.pivot_table(


color
D    3806.532106
E    3683.171591
F    4097.519455
G    4163.998740
H    3846.065529
I    3808.070973
J    3733.765935
dtype: float64

In [92]:
d = diamonds.set_index('cut')
d.loc['Ideal']
d.loc[1]

KeyError: 1

---
## Level 3 — Aggregation

### Task 3: tips — pipe chain, your functions

Load `sns.load_dataset('tips')`. Convert `day`, `time`, and `smoker` to `category`.

Write a `.pipe()` chain with **3 functions** — no starter functions provided, write them yourself:
- One adds `tip_pct` (tip as a percentage of total_bill)
- One adds a group-level column using `transform` — your choice of grouping and metric
- One classifies rows into at least 3 tiers using `.apply()`

After the chain:
- `groupby(['day', 'time'], observed=True)` — which `(day, time)` combination has the highest mean `tip_pct`?
- Do smokers or non-smokers tip a higher percentage on average? Use `groupby` with `observed=True`.

In [67]:
# Your code here

tips = sns.load_dataset('tips')
tips[['day','time','smoker']] = tips[['day','time','smoker']].astype('category')

def add_pct(df):
    df['tip_pct'] = df['tip']/df['total_bill']
    return df
def add_sex_tip_avg(df):
    df['sex_tip_avg'] = df.groupby('sex', observed = True)['tip_pct'].transform('mean')
    return df
def flag_big(df):
    df['meal_size'] = df.apply(lambda row: 'big' if row['total_bill']>50
                               else 'mid' if row['total_bill']>30
                               else 'small', axis =1)
    return df 

    
res = (
    tips.copy()
    .pipe(add_pct)
    .pipe(add_sex_tip_avg)
    .pipe(flag_big)
)

g = res.groupby(['day','time'], observed=True)['tip_pct'].mean()
g.idxmax()

res.groupby('smoker', observed=True)['tip_pct'].mean().idxmax()

'Yes'

---
## Level 4 — Real-world

### Task 4: titanic — survival audit

Load `sns.load_dataset('titanic')`. No scaffolding — use the tools you know.

Requirements:
- Define an ordered `CategoricalDtype` for `pclass` (`3 < 2 < 1`). Filter to passengers in the **top two classes** using an ordered comparison (no `.isin()`).
- On that filtered set, build a `.pipe()` chain that adds at least two new columns of your choice.
- Produce a pivot table showing survival rate by `pclass` × `sex`. Set a `(pclass, sex)` MultiIndex on the result and use `.xs()` to extract one slice.
- One question to answer with NumPy: among passengers 60+, what is the survival rate? Compare it to the overall rate.

In [93]:
# Your code here

titanic = sns.load_dataset('titanic')
p = pd.CategoricalDtype(categories=[3,2,1],
                        ordered=True)

titanic['pclass'] = titanic['pclass'].astype(p)

f = titanic[titanic['pclass']>=2].copy()
def add_senior(df):
    df['is_senior'] = df['age']>60
    return df 
def add_highfare(df):
    df['is_high'] = df['fare']>np.median(df['fare'])
    return df
res = (
    f
    .pipe(add_senior)
    .pipe(add_highfare)
)

p= res.pivot_table(
    index = 'pclass',
    columns = 'sex',
    values = 'survived',
    observed = True
)

pm = p.stack().to_frame('surv_rate')
pm.xs(2,level='pclass').xs('female')

np.mean(titanic[titanic['age']>=60]['survived'])
np.mean(titanic['survived'])


np.float64(0.3838383838383838)

In [94]:
pm.xs((2,'female'))

surv_rate    0.921053
Name: (2, female), dtype: float64